# 06 — Multi-Model Comparison

Benchmarks additional classifiers and anomaly detectors against BehaviorDNA's current models
using the same train/val/test splits.

**Identification (multi-class player recognition):**

| Model | Notes |
|---|---|
| LightGBM | Current production model |
| RandomForest | Ensemble, no boosting |
| XGBoost | Gradient boosting, different regularisation |
| SVC (RBF) | Kernel SVM, strong on small datasets |

**Detection (bot / anomaly scoring, unsupervised):**

| Model | Notes |
|---|---|
| IsolationForest | Current production model |
| LocalOutlierFactor | Density-based, novelty=True |
| One-Class SVM | Kernel boundary around normal behaviour |

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC, OneClassSVM
from xgboost import XGBClassifier

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from pipeline.features.run import FEATURE_COLS

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

SPLITS = ROOT / "data" / "splits"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

In [ ]:
train_df = pd.read_parquet(SPLITS / "train.parquet")
val_df   = pd.read_parquet(SPLITS / "val.parquet")
test_df  = pd.read_parquet(SPLITS / "test.parquet")

print(f"Train: {len(train_df)} windows, {train_df['player'].nunique()} players")
print(f"Val  : {len(val_df)} windows")
print(f"Test : {len(test_df)} windows")

# Combine train+val for final fit before test evaluation
trainval_df = pd.concat([train_df, val_df], ignore_index=True)

def prep(df):
    return df[FEATURE_COLS].fillna(0.0).values

scaler = StandardScaler()
X_train = scaler.fit_transform(prep(trainval_df))
X_test  = scaler.transform(prep(test_df))

le = LabelEncoder()
y_train = le.fit_transform(trainval_df["player"])
y_test  = le.transform(test_df["player"])

print("\nClasses:", list(le.classes_))

---
## Part 1 — Identification Models

In [ ]:
id_models = {
    "LightGBM": LGBMClassifier(
        num_leaves=63, learning_rate=0.05, n_estimators=300,
        min_child_samples=1, class_weight="balanced", verbose=-1,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        use_label_encoder=False, eval_metric="mlogloss",
        verbosity=0, random_state=42,
    ),
    "SVC (RBF)": SVC(
        kernel="rbf", class_weight="balanced", probability=True, random_state=42,
    ),
}

id_results = []
fitted_models = {}

for name, model in id_models.items():
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    id_results.append({"model": name, "accuracy": acc, "f1_weighted": f1, "fit_s": fit_time})
    fitted_models[name] = (model, y_pred)
    print(f"{name:20s}  acc={acc:.3f}  f1={f1:.3f}  fit={fit_time:.1f}s")

id_df = pd.DataFrame(id_results).sort_values("accuracy", ascending=False)
id_df

In [ ]:
# Accuracy + F1 bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ["accuracy", "f1_weighted"]):
    sorted_df = id_df.sort_values(metric, ascending=False)
    bars = ax.bar(sorted_df["model"], sorted_df[metric],
                  color=sns.color_palette("muted", len(id_df)))
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(metric)
    ax.set_title(f"Identification — {metric}")
    ax.tick_params(axis="x", rotation=15)
    for bar, val in zip(bars, sorted_df[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "model_comparison_identification.png", bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrix grid
n = len(fitted_models)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]

for ax, (name, (_, y_pred)) in zip(axes, fitted_models.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=10)
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Identification — Confusion Matrices (test set)", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / "confusion_matrices_comparison.png", bbox_inches="tight")
plt.show()

---
## Part 2 — Detection / Anomaly Models

All three models are trained on the full training set (no labels needed) and scored on the test set.
Lower scores = more anomalous for IsolationForest and LOF; higher decision function = more anomalous for OneClassSVM.

In [ ]:
det_models = {
    "IsolationForest": IsolationForest(
        n_estimators=200, contamination=0.05, random_state=42,
    ),
    "LocalOutlierFactor": LocalOutlierFactor(
        n_neighbors=20, contamination=0.05, novelty=True,
    ),
    "OneClassSVM": OneClassSVM(
        kernel="rbf", nu=0.05,
    ),
}

det_results = []
det_scores  = {}

for name, model in det_models.items():
    t0 = time.perf_counter()
    model.fit(X_train)
    fit_time = time.perf_counter() - t0

    scores = model.score_samples(X_test)
    preds  = model.predict(X_test)
    pct_outlier = (preds == -1).mean()

    det_results.append({
        "model": name,
        "mean_score": scores.mean(),
        "std_score": scores.std(),
        "pct_outlier": pct_outlier,
        "fit_s": fit_time,
    })
    det_scores[name] = scores
    print(f"{name:22s}  mean={scores.mean():.4f}  pct_outlier={pct_outlier:.1%}  fit={fit_time:.1f}s")

det_df = pd.DataFrame(det_results)
det_df

In [ ]:
# Anomaly score distributions — violin plot
score_data = pd.DataFrame(det_scores)
score_long = score_data.melt(var_name="model", value_name="score")

fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=score_long, x="model", y="score",
               palette="muted", inner="quartile", ax=ax)
ax.set_title("Detection Models — Anomaly Score Distributions (test set)")
ax.set_xlabel("")
ax.set_ylabel("score_samples()  (lower = more anomalous)")
plt.tight_layout()
plt.savefig(FIGURES / "anomaly_score_distributions.png", bbox_inches="tight")
plt.show()

---
## Final Summary

In [ ]:
print("=== Identification Results ===")
print(id_df.to_string(index=False, float_format="{:.4f}".format))

print("\n=== Detection Results ===")
print(det_df.to_string(index=False, float_format="{:.4f}".format))

### Recommendations for Pipeline Promotion

*(Fill in after running — update based on actual results)*

**Identification:**
- Promote the highest-accuracy model that is also fast to train (LightGBM is a strong baseline for small datasets).
- XGBoost is a good second candidate if accuracy is higher and training time acceptable.
- SVC scales poorly with dataset size — only worth promoting if n_windows stays small.

**Detection:**
- IsolationForest is already in production and fast.
- LocalOutlierFactor handles local density variations — worth testing if IsolationForest shows high false-positive rate.
- OneClassSVM is slowest; use only if a hard boundary is needed.

**Next steps:** add winning model types to `configs/training.yaml` model.type options and wire them into `pipeline/training/run.py`.